
# ReliableVAEP
## Premier League 2015/16 PCS/RAR Stress Test
### Fully standalone Kaggle notebook

This notebook tests PCS/RAR on the **large, previously unstable Premier League 2015/16 leaderboard**.

It is designed to answer two different questions:

### 1. Does PCS/RAR actually change the static VAEP ranking?

The notebook recomputes canonical VAEP from StatsBomb open data, applies the same eligibility rule used in the earlier PL analysis, and produces one full table containing:

- VAEP/90
- original VAEP rank
- bootstrap median rank
- 95% bootstrap rank interval
- 90% supported rank band
- Pairwise Consensus Score (PCS)
- Reliability-Adjusted Rank (RAR)
- change from original VAEP rank to RAR

It explicitly prints both:

- **Top players by original VAEP rank**
- **Top players by RAR**

and a third table showing the **largest rank changes**.

### 2. If RAR changes the ranking, is it actually more reproducible?

A changed ranking is not automatically better.

For a fixed cohort of players with at least 30 observed matches, the notebook repeatedly creates two non-overlapping match samples and compares:

\[
\rho_{\mathrm{VAEP}}
=
\operatorname{Spearman}
(R^{VAEP}_A,R^{VAEP}_B)
\]

against

\[
\rho_{\mathrm{RAR}}
=
\operatorname{Spearman}
(R^{RAR}_A,R^{RAR}_B)
\]

at \(n=5,10,15\) matches per independent sample.

The key quantity is:

\[
\Delta\rho
=
\rho_{\mathrm{RAR}}-\rho_{\mathrm{VAEP}}
\]

A consistently positive value would support the claim that RAR is more reproducible.

---

## Important

The earlier PL study produced roughly **325 eligible players** and a median bootstrap rank-interval width of about **52 ranks**. Those are reference values only.

This notebook recomputes the entire Premier League analysis using the canonical VAEP pipeline. It does **not** force the eligible-player count, interval width, or ranking to match the earlier result.

---

## Kaggle instructions

1. Open **Settings**
2. Turn **Internet = On**
3. Run all cells from the top

No earlier ReliableVAEP notebook or intermediate CSV/NPY file is required.


> **Canonical ReliableVAEP bootstrap setting for the final SSAC analysis:** `FULL_BOOTSTRAP_B = 3500`.
>
> The value is not an arbitrary round number. ReliableVAEP treats a pairwise ordering as supported at
> \(P_{ij}\ge 0.90\) (or \(\le 0.10\)). Requiring a 95% Wilson Monte Carlo half-width of at most
> 0.01 at \(p=0.90\) gives a minimum of 3,461 resamples. The final analyses therefore use 3,500
> player-preserving match-level bootstrap draws, the smallest 100-draw increment above that bound.
> The completed cross-seed sensitivity test confirmed very high rank-order and pair-classification
> stability at \(B=3500\) across the study datasets.
>
> **Important:** `INNER_BOOTSTRAP_B` values used inside repeated split-sample validation are left at
> their original values. They are nested computational approximations for hundreds of repeated
> validation runs and are not used to produce the paper's primary full-data rank intervals or
> 0.90 pairwise-support decisions.

In [1]:
# FINAL SSAC RUN CONFIGURATION CHECK
CANONICAL_FULL_BOOTSTRAP_B = 3500
print("ReliableVAEP final SSAC run")
print("Canonical full-data bootstrap draws:", CANONICAL_FULL_BOOTSTRAP_B)
print("Run this notebook from top to bottom and use only the newly generated outputs.")


ReliableVAEP final SSAC run
Canonical full-data bootstrap draws: 3500
Run this notebook from top to bottom and use only the newly generated outputs.


In [2]:

# ================================================================
# CELL 1 — KAGGLE-SAFE ISOLATED ENVIRONMENT
# ================================================================
#
# IMPORTANT:
# Kaggle Internet must be ON.
#
# This setup:
#   1. uses /kaggle/working only;
#   2. installs only the small pure-Python "virtualenv" helper into
#      Kaggle's host environment if it is missing;
#   3. creates /kaggle/working/rv_vaep_env;
#   4. installs the complete VAEP stack INSIDE that environment;
#   5. leaves Kaggle's NumPy/pandas/scipy/sklearn untouched.
#
# No runtime restart is required.
# ================================================================

import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working").exists():
    raise RuntimeError(
        "This notebook is the Kaggle-specific version, but /kaggle/working "
        "was not found. Please run it inside a Kaggle Notebook."
    )

if sys.version_info >= (3, 13):
    raise RuntimeError(
        "Please use a Kaggle Python runtime <= 3.12. "
        "socceraction 1.5.3 is not designed for Python 3.13."
    )

BASE_DIR = Path("/kaggle/working")
ENV_DIR = BASE_DIR / "rv_vaep_env"
ENV_PY = ENV_DIR / "bin" / "python"
ENV_PIP = ENV_DIR / "bin" / "pip"
READY = ENV_DIR / ".reliablevaep_kaggle_ready_v1"

IMPORT_TEST = r"""
import numpy
import pandas
import scipy
import sklearn
import xgboost
import socceraction

from socceraction.data.statsbomb import StatsBombLoader
from socceraction.vaep import VAEP
import socceraction.spadl as spadl

print("numpy        =", numpy.__version__)
print("pandas       =", pandas.__version__)
print("scipy        =", scipy.__version__)
print("sklearn      =", sklearn.__version__)
print("xgboost      =", xgboost.__version__)
print("socceraction =", getattr(socceraction, "__version__", "installed"))

# NumPy/pandas binary compatibility smoke test.
df = pandas.DataFrame({"x": numpy.arange(5, dtype=float)})
assert float(df["x"].sum()) == 10.0

print("KAGGLE_ISOLATED_ENV_OK")
"""


def run_capture(args, timeout=None):
    return subprocess.run(
        [str(x) for x in args],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )


def environment_is_healthy():
    if not (
        ENV_PY.exists()
        and ENV_PIP.exists()
        and READY.exists()
    ):
        return False

    test = run_capture(
        [ENV_PY, "-c", IMPORT_TEST]
    )

    if (
        test.returncode == 0
        and "KAGGLE_ISOLATED_ENV_OK" in test.stdout
    ):
        print("Existing isolated environment verified:")
        print(test.stdout)
        return True

    print("Existing isolated environment is incomplete/unhealthy.")
    print(test.stdout)
    return False


if not environment_is_healthy():

    # Remove only our own previous environment.
    if ENV_DIR.exists():
        print("Removing incomplete ReliableVAEP environment:", ENV_DIR)
        shutil.rmtree(ENV_DIR, ignore_errors=True)

    # ------------------------------------------------------------
    # Install virtualenv only if unavailable.
    # virtualenv is pure Python and does not alter NumPy/pandas.
    # ------------------------------------------------------------
    if importlib.util.find_spec("virtualenv") is None:
        print("Installing the lightweight virtualenv helper into Kaggle...")
        helper = run_capture([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            "virtualenv==20.28.0",
        ])

        print(helper.stdout)

        if helper.returncode != 0:
            raise RuntimeError(
                "Could not install virtualenv.\n"
                "Make sure Kaggle Internet is ON.\n\n"
                + helper.stdout
            )
    else:
        print("virtualenv helper is already available.")

    # ------------------------------------------------------------
    # Create isolated environment using virtualenv.
    # This avoids system ensurepip entirely.
    # ------------------------------------------------------------
    print("\nCreating isolated environment:", ENV_DIR)

    create = run_capture([
        sys.executable,
        "-m",
        "virtualenv",
        "--clear",
        str(ENV_DIR),
    ])

    print(create.stdout)

    if (
        create.returncode != 0
        or not ENV_PY.exists()
        or not ENV_PIP.exists()
    ):
        raise RuntimeError(
            "virtualenv could not create the isolated environment.\n\n"
            + create.stdout
        )

    # ------------------------------------------------------------
    # Install one internally consistent stack.
    # These packages exist ONLY inside /kaggle/working/rv_vaep_env.
    # ------------------------------------------------------------
    packages = [
        "numpy==1.26.4",
        "pandas==2.1.4",
        "scipy==1.13.1",
        "scikit-learn==1.5.2",
        "xgboost==2.1.4",
        "matplotlib==3.9.2",
        "tqdm==4.66.5",
        "multimethod==1.10",
        "pandera==0.17.2",
        "socceraction[statsbomb]==1.5.3",
    ]

    print("\nInstalling the isolated ReliableVAEP stack...")
    print("This can take a few minutes on the first run.")

    install = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        *packages,
    ])

    print(install.stdout)

    if install.returncode != 0:
        raise RuntimeError(
            "Package installation failed INSIDE the isolated environment.\n"
            "Kaggle's host NumPy/pandas environment has not been changed.\n\n"
            + install.stdout
        )

    # ------------------------------------------------------------
    # Verify dependency consistency.
    # ------------------------------------------------------------
    print("\nChecking isolated dependency consistency...")

    check = run_capture([
        ENV_PY,
        "-m",
        "pip",
        "check",
    ])

    print(
        check.stdout
        if check.stdout.strip()
        else "pip check: no dependency conflicts"
    )

    if check.returncode != 0:
        raise RuntimeError(
            "Dependency conflicts remain inside the isolated environment.\n\n"
            + check.stdout
        )

    # ------------------------------------------------------------
    # Import and binary compatibility test.
    # ------------------------------------------------------------
    print("\nVerifying isolated imports...")

    verify = run_capture([
        ENV_PY,
        "-c",
        IMPORT_TEST,
    ])

    print(verify.stdout)

    if (
        verify.returncode != 0
        or "KAGGLE_ISOLATED_ENV_OK" not in verify.stdout
    ):
        raise RuntimeError(
            "The isolated environment was created, but its scientific stack "
            "failed verification.\n\n"
            + verify.stdout
        )

    READY.write_text("ready", encoding="utf-8")

    print("\nKaggle isolated environment setup completed successfully.")

else:
    print("\nReusing the verified Kaggle isolated environment.")

print("\nAnalysis Python:", ENV_PY)
print("Kaggle host scientific packages were NOT downgraded or replaced.")


Installing the lightweight virtualenv helper into Kaggle...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 119.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 289.0 MB/s eta 0:00:00


Creating isolated environment: /kaggle/working/rv_vaep_env
created virtual environment CPython3.12.13.final.0-64 in 340ms
  creator CPython3Posix(dest=/kaggle/working/rv_vaep_env, clear=True, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.local/share/virtualenv)
    added seed packages: pip==24.3.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator


Installing the isolated ReliableVAEP stack...
This can take a few minutes on the first run.
Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
INFO: pip is looking at multiple versions of contourpy to determine which version is compatible with other

In [3]:

# ================================================================
# CELL 2 — WRITE THE COMPLETE PREMIER LEAGUE ANALYSIS
# ================================================================

from pathlib import Path

SCRIPT_PATH = Path("/kaggle/working/reliablevaep_pl1516_pcs_rar_stress_test.py")

ANALYSIS_SCRIPT = '\n# ================================================================\n# ReliableVAEP — Premier League 2015/16 PCS/RAR Stress Test\n# FULLY STANDALONE KAGGLE ANALYSIS\n# ================================================================\n\nimport json\nimport os\nimport shutil\nimport time\nimport urllib.request\nimport warnings\nfrom pathlib import Path\n\n# Kaggle exports a notebook-only matplotlib backend.  The analysis\n# runs in an isolated subprocess, so force a normal file backend first.\nos.environ["MPLBACKEND"] = "Agg"\n\nimport matplotlib\nmatplotlib.use("Agg", force=True)\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport xgboost as xgb\n\nfrom scipy.optimize import minimize\nfrom scipy.special import expit, logit\nfrom scipy.stats import spearmanr, wilcoxon\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score\nfrom sklearn.model_selection import GroupKFold, GroupShuffleSplit\nfrom tqdm.auto import tqdm\n\nfrom socceraction.data.statsbomb import StatsBombLoader\nimport socceraction.spadl as spadl\nfrom socceraction.vaep import VAEP\nfrom socceraction.vaep import formula as vaep_formula\n\nwarnings.filterwarnings("ignore")\n\n# ----------------------------------------------------------------\n# Reproducibility and analysis settings\n# ----------------------------------------------------------------\nSEED = 20260905\nnp.random.seed(SEED)\n\nCOMPETITION_ID = 2\nSEASON_ID = 27\nDATASET_LABEL = "Premier League 2015/16"\n\n# Same eligibility rule used in the earlier ReliableVAEP PL study.\nMIN_MATCHES = 5\nMIN_MINUTES = 300\nMIN_ACTIONS = 150\n\n# Full static ReliableVAEP / PCS / RAR analysis.\nFULL_BOOTSTRAP_B = 3500\nPAIRWISE_SUPPORT = 0.90\nBT_RIDGE = 1e-4\n\n# Independent reproducibility stress test.\n# Fixed cohort: players with at least 30 observed matches.\n# Therefore n=5, 10, 15 can all be tested with non-overlapping samples.\nVALIDATION_MIN_MATCHES = 30\nVALIDATION_N_VALUES = [5, 10, 15]\nVALIDATION_REPS = 200\nINNER_BOOTSTRAP_B = 300\n\nOUTER_FOLDS = 5\nCALIBRATION_MATCH_FRACTION = 0.20\n\nROOT = Path("/kaggle/working")\nCACHE_ROOT = ROOT / "rv_pl1516_cache"\nOUT_ROOT = ROOT / "rv_pl1516_pcs_rar_outputs"\n\nCACHE_ROOT.mkdir(parents=True, exist_ok=True)\nOUT_ROOT.mkdir(parents=True, exist_ok=True)\n\nBASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"\n\nprint("=" * 96)\nprint("ReliableVAEP Premier League 2015/16 PCS/RAR stress test")\nprint("=" * 96)\nprint("Eligibility:", f">={MIN_MATCHES} matches, >={MIN_MINUTES} minutes, >={MIN_ACTIONS} actions")\nprint("Full-data bootstrap draws:", FULL_BOOTSTRAP_B)\nprint("Pairwise support threshold:", PAIRWISE_SUPPORT)\nprint("Validation fixed cohort: >=", VALIDATION_MIN_MATCHES, "matches")\nprint("Validation n values:", VALIDATION_N_VALUES)\nprint("Validation repetitions per n:", VALIDATION_REPS)\nprint()\nprint(\n    "Reference only: the earlier PL analysis had about 325 eligible players and "\n    "a median 95% bootstrap rank-interval width of about 52. "\n    "This notebook recomputes everything canonically and does NOT force those values."\n)\nprint()\n\n\n# ================================================================\n# Utility functions\n# ================================================================\n\ndef get_json(url, path, retries=5):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n\n    if path.exists():\n        try:\n            with open(path, encoding="utf-8") as f:\n                return json.load(f)\n        except Exception:\n            path.unlink(missing_ok=True)\n\n    for attempt in range(1, retries + 1):\n        try:\n            req = urllib.request.Request(\n                url,\n                headers={"User-Agent": "ReliableVAEP-PL1516/1.0"},\n            )\n            with urllib.request.urlopen(req, timeout=120) as response:\n                obj = json.loads(response.read().decode("utf-8"))\n            with open(path, "w", encoding="utf-8") as f:\n                json.dump(obj, f)\n            return obj\n        except Exception as exc:\n            if attempt == retries:\n                raise RuntimeError(f"Download failed: {url}") from exc\n            time.sleep(min(2 ** attempt, 15))\n\n\ndef make_model(seed):\n    return xgb.XGBClassifier(\n        n_estimators=180,\n        max_depth=3,\n        learning_rate=0.05,\n        min_child_weight=5,\n        subsample=0.85,\n        colsample_bytree=0.85,\n        reg_lambda=2.0,\n        objective="binary:logistic",\n        eval_metric="logloss",\n        tree_method="hist",\n        random_state=seed,\n        n_jobs=-1,\n    )\n\n\ndef fit_platt(prob, y):\n    prob = np.asarray(prob, dtype=float)\n    y = np.asarray(y, dtype=int)\n    if len(np.unique(y)) < 2:\n        return None\n    z = logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1)\n    model = LogisticRegression(solver="lbfgs", max_iter=1000)\n    model.fit(z, y)\n    return model\n\n\ndef apply_platt(model, prob):\n    prob = np.asarray(prob, dtype=float)\n    if model is None:\n        return prob.copy()\n    z = logit(np.clip(prob, 1e-6, 1 - 1e-6)).reshape(-1, 1)\n    return model.predict_proba(z)[:, 1]\n\n\ndef safe_auc(y, p):\n    return float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan\n\n\ndef baseline_brier(y):\n    y = np.asarray(y, dtype=float)\n    return float(np.mean((y - y.mean()) ** 2))\n\n\ndef vaep90(q):\n    minutes = float(q["minutes_played"].sum())\n    if minutes <= 0:\n        return np.nan\n    return float(90.0 * q["vaep"].sum() / minutes)\n\n\ndef ranks_desc(values):\n    return (\n        pd.Series(np.asarray(values, dtype=float))\n        .rank(ascending=False, method="average")\n        .to_numpy(dtype=float)\n    )\n\n\ndef player_id_to_jsonable(x):\n    if isinstance(x, (np.integer, int)):\n        return int(x)\n    try:\n        xf = float(x)\n        if xf.is_integer():\n            return int(xf)\n        return xf\n    except Exception:\n        return str(x)\n\n\nSBL = StatsBombLoader(getter="remote", creds=None)\n\n\n# ================================================================\n# STEP 1 — Canonical Premier League 2015/16 player-match VAEP\n# ================================================================\n\ndef build_pl_player_match():\n    dataset_dir = OUT_ROOT / "canonical_build"\n    dataset_dir.mkdir(parents=True, exist_ok=True)\n\n    marker = dataset_dir / "BUILD_COMPLETE.json"\n    pm_pickle = dataset_dir / "player_match_vaep.pkl"\n    cohort_csv = dataset_dir / "eligible_cohort.csv"\n    metrics_csv = dataset_dir / "crossfit_model_metrics.csv"\n\n    if marker.exists() and pm_pickle.exists() and cohort_csv.exists() and metrics_csv.exists():\n        try:\n            meta = json.loads(marker.read_text(encoding="utf-8"))\n            pm = pd.read_pickle(pm_pickle)\n            cohort = pd.read_csv(cohort_csv)\n            metrics = pd.read_csv(metrics_csv)\n            print("Loading completed cached canonical PL build.")\n            print("Matches:", meta["matches"])\n            print("Eligible players:", meta["eligible_players"])\n            return pm, cohort, metrics, meta\n        except Exception as exc:\n            print("Cached build could not be loaded; rebuilding:", exc)\n\n    # ------------------------------------------------------------\n    # Match list\n    # ------------------------------------------------------------\n    match_file = CACHE_ROOT / "matches" / f"{COMPETITION_ID}_{SEASON_ID}.json"\n    matches_raw = get_json(\n        f"{BASE_URL}/matches/{COMPETITION_ID}/{SEASON_ID}.json",\n        match_file,\n    )\n\n    games = pd.DataFrame([\n        {\n            "game_id": int(m["match_id"]),\n            "game_date": pd.to_datetime(m["match_date"]),\n            "home_team_id": int(m["home_team"]["home_team_id"]),\n            "away_team_id": int(m["away_team"]["away_team_id"]),\n            "home_team_name": str(m["home_team"]["home_team_name"]),\n            "away_team_name": str(m["away_team"]["away_team_name"]),\n        }\n        for m in matches_raw\n    ]).sort_values(["game_date", "game_id"]).reset_index(drop=True)\n\n    print("StatsBomb PL matches found:", games["game_id"].nunique())\n\n    if games["game_id"].nunique() < 300:\n        raise RuntimeError(\n            "Unexpectedly few Premier League 2015/16 matches were found. "\n            "Check competition/season IDs or StatsBomb availability."\n        )\n\n    games.to_csv(dataset_dir / "game_list.csv", index=False)\n\n    # ------------------------------------------------------------\n    # SPADL actions + official player minutes\n    # ------------------------------------------------------------\n    store = {}\n\n    for g in tqdm(\n        games.itertuples(index=False),\n        total=len(games),\n        desc="PL 2015/16: SPADL",\n        unit="match",\n    ):\n        gid = int(g.game_id)\n\n        action_path = CACHE_ROOT / "spadl" / f"{gid}.pkl"\n        player_path = CACHE_ROOT / "players" / f"{gid}.pkl"\n        action_path.parent.mkdir(parents=True, exist_ok=True)\n        player_path.parent.mkdir(parents=True, exist_ok=True)\n\n        if action_path.exists():\n            actions = pd.read_pickle(action_path)\n        else:\n            events = SBL.events(game_id=gid)\n            actions = (\n                spadl.statsbomb.convert_to_actions(\n                    events=events,\n                    home_team_id=int(g.home_team_id),\n                )\n                .reset_index(drop=True)\n            )\n            actions.to_pickle(action_path)\n\n        if player_path.exists():\n            players = pd.read_pickle(player_path)\n        else:\n            players = SBL.players(game_id=gid)\n            players.to_pickle(player_path)\n\n        if len(actions) == 0:\n            raise RuntimeError(f"No SPADL actions in match {gid}.")\n\n        store[gid] = {\n            "game": pd.Series({\n                "game_id": gid,\n                "home_team_id": int(g.home_team_id),\n                "away_team_id": int(g.away_team_id),\n                "game_date": g.game_date,\n            }),\n            "actions": actions,\n            "players": players,\n        }\n\n    # ------------------------------------------------------------\n    # Canonical VAEP features and labels\n    # ------------------------------------------------------------\n    template = VAEP()\n\n    X_parts = []\n    Y_parts = []\n    row_parts = []\n\n    for gid, item in tqdm(store.items(), desc="Canonical VAEP features", unit="match"):\n        Xg = template.compute_features(\n            item["game"],\n            item["actions"],\n        ).reset_index(drop=True)\n\n        Yg = template.compute_labels(\n            item["game"],\n            item["actions"],\n        ).reset_index(drop=True)\n\n        if len(Xg) != len(item["actions"]) or len(Yg) != len(item["actions"]):\n            raise RuntimeError(f"Feature/label/action alignment failed in match {gid}.")\n\n        X_parts.append(Xg)\n        Y_parts.append(Yg)\n        row_parts.append(\n            pd.DataFrame({\n                "game_id": gid,\n                "action_row": np.arange(len(Xg)),\n            })\n        )\n\n    X = pd.concat(X_parts, ignore_index=True)\n    Y = pd.concat(Y_parts, ignore_index=True)\n    ROW = pd.concat(row_parts, ignore_index=True)\n\n    # Convert any nonnumeric encoded feature columns to numeric forms.\n    for col in X.columns:\n        if pd.api.types.is_bool_dtype(X[col]):\n            X[col] = X[col].astype(np.uint8)\n        elif isinstance(X[col].dtype, pd.CategoricalDtype):\n            X[col] = X[col].cat.codes.astype(np.int32)\n        elif pd.api.types.is_object_dtype(X[col]):\n            X[col] = pd.factorize(X[col].astype(str))[0].astype(np.int32)\n\n    bad_features = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]\n    if bad_features:\n        raise TypeError(f"Non-numeric VAEP feature columns remain: {bad_features}")\n\n    print(\n        f"Action rows={len(X):,}, features={X.shape[1]}, "\n        f"score rate={Y[\'scores\'].mean():.4f}, concede rate={Y[\'concedes\'].mean():.4f}"\n    )\n\n    # ------------------------------------------------------------\n    # Match-level out-of-fold prediction + inner calibration\n    # ------------------------------------------------------------\n    groups = ROW["game_id"].to_numpy()\n    unique_matches = np.unique(groups)\n    n_folds = min(OUTER_FOLDS, len(unique_matches))\n\n    outer = GroupKFold(n_splits=n_folds)\n\n    oof = {\n        "scores_raw": np.full(len(X), np.nan),\n        "scores_cal": np.full(len(X), np.nan),\n        "concedes_raw": np.full(len(X), np.nan),\n        "concedes_cal": np.full(len(X), np.nan),\n    }\n\n    metric_rows = []\n\n    for fold, (train_idx, test_idx) in enumerate(\n        outer.split(X, groups=groups),\n        start=1,\n    ):\n        inner = GroupShuffleSplit(\n            n_splits=1,\n            test_size=CALIBRATION_MATCH_FRACTION,\n            random_state=SEED + fold,\n        )\n\n        fit_rel, cal_rel = next(\n            inner.split(\n                X.iloc[train_idx],\n                groups=groups[train_idx],\n            )\n        )\n\n        fit_idx = train_idx[fit_rel]\n        cal_idx = train_idx[cal_rel]\n\n        row = {\n            "fold": fold,\n            "fit_matches": len(np.unique(groups[fit_idx])),\n            "calibration_matches": len(np.unique(groups[cal_idx])),\n            "test_matches": len(np.unique(groups[test_idx])),\n        }\n\n        for target in ["scores", "concedes"]:\n            model = make_model(\n                SEED + fold * 10 + (1 if target == "scores" else 2)\n            )\n\n            model.fit(\n                X.iloc[fit_idx],\n                Y.iloc[fit_idx][target].astype(int),\n            )\n\n            p_cal_raw = model.predict_proba(X.iloc[cal_idx])[:, 1]\n            calibrator = fit_platt(\n                p_cal_raw,\n                Y.iloc[cal_idx][target].astype(int),\n            )\n\n            p_test_raw = model.predict_proba(X.iloc[test_idx])[:, 1]\n            p_test_cal = apply_platt(calibrator, p_test_raw)\n\n            oof[f"{target}_raw"][test_idx] = p_test_raw\n            oof[f"{target}_cal"][test_idx] = p_test_cal\n\n            y_test = Y.iloc[test_idx][target].astype(int).to_numpy()\n            base = baseline_brier(y_test)\n\n            row[f"{target}_auc"] = safe_auc(y_test, p_test_cal)\n            row[f"{target}_ap"] = float(average_precision_score(y_test, p_test_cal))\n            row[f"{target}_brier_baseline"] = base\n            row[f"{target}_brier_raw"] = float(brier_score_loss(y_test, p_test_raw))\n            row[f"{target}_brier_cal"] = float(brier_score_loss(y_test, p_test_cal))\n            row[f"{target}_brier_skill_cal"] = (\n                1.0 - row[f"{target}_brier_cal"] / base if base > 0 else np.nan\n            )\n\n        metric_rows.append(row)\n\n    for key, values in oof.items():\n        if np.isnan(values).any():\n            raise RuntimeError(f"Incomplete OOF predictions: {key}")\n\n    metrics = pd.DataFrame(metric_rows)\n    metrics.to_csv(metrics_csv, index=False)\n\n    print("\\nCross-fit model means:")\n    print(\n        metrics.mean(numeric_only=True)\n        .round(4)\n        .to_string()\n    )\n\n    # ------------------------------------------------------------\n    # Canonical action VAEP\n    # ------------------------------------------------------------\n    scored_parts = []\n    cursor = 0\n\n    for gid, item in tqdm(store.items(), desc="Canonical action VAEP", unit="match"):\n        actions = item["actions"].reset_index(drop=True)\n        n_actions = len(actions)\n\n        ps = pd.Series(oof["scores_cal"][cursor:cursor + n_actions])\n        pc = pd.Series(oof["concedes_cal"][cursor:cursor + n_actions])\n\n        named = spadl.add_names(actions.copy()).reset_index(drop=True)\n        values = vaep_formula.value(named, ps, pc).reset_index(drop=True)\n\n        z = named.copy()\n        z["game_id"] = gid\n        z["game_date"] = item["game"]["game_date"]\n        z["p_score"] = ps.to_numpy()\n        z["p_concede"] = pc.to_numpy()\n\n        for col in ["offensive_value", "defensive_value", "vaep_value"]:\n            z[col] = values[col].to_numpy()\n\n        scored_parts.append(z)\n        cursor += n_actions\n\n    if cursor != len(X):\n        raise RuntimeError("OOF probability/action alignment failed.")\n\n    scored = pd.concat(scored_parts, ignore_index=True)\n\n    # ------------------------------------------------------------\n    # Goal sanity check\n    # ------------------------------------------------------------\n    goal_mask = (\n        scored["type_name"].isin(["shot", "shot_freekick", "shot_penalty"])\n        & scored["result_name"].eq("success")\n    )\n\n    if int(goal_mask.sum()) == 0:\n        raise RuntimeError("No successful goals detected for VAEP sanity checking.")\n\n    goal_values = scored.loc[goal_mask, "vaep_value"]\n    median_goal_vaep = float(goal_values.median())\n    positive_goal_fraction = float((goal_values > 0).mean())\n\n    print(\n        "\\nGoal-value sanity check:",\n        f"n={int(goal_mask.sum())}, "\n        f"median successful-goal VAEP={median_goal_vaep:.4f}, "\n        f"positive fraction={positive_goal_fraction:.3f}"\n    )\n\n    if median_goal_vaep <= 0:\n        raise RuntimeError(\n            "Canonical VAEP sanity check failed: median successful-goal value is not positive."\n        )\n\n    # ------------------------------------------------------------\n    # Player-match aggregation for ALL teams\n    # ------------------------------------------------------------\n    pm_parts = []\n\n    keeper_types = {\n        "keeper_save",\n        "keeper_claim",\n        "keeper_punch",\n        "keeper_pick_up",\n    }\n\n    keeper_action_ids = set(\n        scored.loc[\n            scored["type_name"].isin(keeper_types),\n            "player_id",\n        ]\n        .dropna()\n        .tolist()\n    )\n\n    for gid, item in tqdm(store.items(), desc="Player-match VAEP", unit="match"):\n        players = item["players"].copy()\n        players = players[\n            players["minutes_played"].fillna(0) > 0\n        ].copy()\n\n        ga = scored[scored["game_id"] == gid].copy()\n\n        agg = (\n            ga.dropna(subset=["player_id"])\n            .groupby("player_id", as_index=False)\n            .agg(\n                vaep=("vaep_value", "sum"),\n                offensive_value=("offensive_value", "sum"),\n                defensive_value=("defensive_value", "sum"),\n                actions=("action_id", "size"),\n            )\n        )\n\n        goals = (\n            ga[\n                ga["type_name"].isin(["shot", "shot_freekick", "shot_penalty"])\n                & ga["result_name"].eq("success")\n            ]\n            .dropna(subset=["player_id"])\n            .groupby("player_id")\n            .size()\n            .rename("goals")\n            .reset_index()\n        )\n\n        agg = agg.merge(goals, on="player_id", how="left")\n        agg["goals"] = agg["goals"].fillna(0).astype(int)\n\n        q = players.merge(agg, on="player_id", how="left")\n\n        for col in ["vaep", "offensive_value", "defensive_value"]:\n            q[col] = q[col].fillna(0.0)\n\n        for col in ["actions", "goals"]:\n            q[col] = q[col].fillna(0).astype(int)\n\n        q["game_id"] = gid\n        q["game_date"] = item["game"]["game_date"]\n        q["vaep90_match"] = (\n            90.0 * q["vaep"] / q["minutes_played"]\n        )\n\n        if "starting_position_name" in q.columns:\n            q["is_goalkeeper_position"] = (\n                q["starting_position_name"]\n                .astype(str)\n                .str.contains("Goalkeeper", case=False, na=False)\n            )\n        else:\n            q["is_goalkeeper_position"] = False\n\n        q["is_goalkeeper"] = (\n            q["is_goalkeeper_position"].astype(bool)\n            | q["player_id"].isin(keeper_action_ids)\n        )\n\n        pm_parts.append(q)\n\n    pm = pd.concat(pm_parts, ignore_index=True)\n\n    # ------------------------------------------------------------\n    # Eligibility\n    # ------------------------------------------------------------\n    cohort = (\n        pm.groupby(["player_id", "player_name"], as_index=False)\n        .agg(\n            total_matches=("game_id", "nunique"),\n            total_minutes=("minutes_played", "sum"),\n            total_actions=("actions", "sum"),\n            total_vaep=("vaep", "sum"),\n            offensive_value=("offensive_value", "sum"),\n            defensive_value=("defensive_value", "sum"),\n            goals=("goals", "sum"),\n            goalkeeper_match_rows=("is_goalkeeper", "sum"),\n        )\n    )\n\n    cohort["vaep90"] = (\n        90.0 * cohort["total_vaep"] / cohort["total_minutes"]\n    )\n\n    cohort["is_goalkeeper"] = cohort["goalkeeper_match_rows"] > 0\n\n    cohort = cohort[\n        (cohort["total_matches"] >= MIN_MATCHES)\n        & (cohort["total_minutes"] >= MIN_MINUTES)\n        & (cohort["total_actions"] >= MIN_ACTIONS)\n    ].copy()\n\n    cohort = (\n        cohort.sort_values("vaep90", ascending=False)\n        .reset_index(drop=True)\n    )\n\n    cohort["raw_VAEP_rank"] = np.arange(1, len(cohort) + 1)\n\n    print("\\nEligible players:", len(cohort))\n    print("Goalkeepers among eligible players:", int(cohort["is_goalkeeper"].sum()))\n\n    if len(cohort) < 200:\n        print(\n            "WARNING: fewer than 200 eligible players were produced. "\n            "The notebook will continue, but compare eligibility with the earlier PL run."\n        )\n\n    pm.to_pickle(pm_pickle)\n    pm.to_csv(dataset_dir / "player_match_vaep.csv", index=False)\n    cohort.to_csv(cohort_csv, index=False)\n\n    meta = {\n        "competition_id": COMPETITION_ID,\n        "season_id": SEASON_ID,\n        "matches": int(games["game_id"].nunique()),\n        "eligible_players": int(len(cohort)),\n        "eligible_goalkeepers": int(cohort["is_goalkeeper"].sum()),\n        "median_successful_goal_vaep": median_goal_vaep,\n        "positive_goal_fraction": positive_goal_fraction,\n    }\n\n    marker.write_text(json.dumps(meta, indent=2), encoding="utf-8")\n\n    return pm, cohort, metrics, meta\n\n\n# ================================================================\n# STEP 2 — Full-data match bootstrap\n# ================================================================\n\ndef bootstrap_full_cohort(pm, cohort, B=FULL_BOOTSTRAP_B, seed=SEED + 100):\n    player_ids = cohort["player_id"].tolist()\n    K = len(player_ids)\n\n    grouped = {\n        pid: q.reset_index(drop=True)\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    V = np.empty((B, K), dtype=float)\n\n    rng = np.random.default_rng(seed)\n\n    for j, pid in enumerate(\n        tqdm(player_ids, desc="Full-cohort match bootstrap", unit="player")\n    ):\n        q = grouped[pid]\n        va = q["vaep"].to_numpy(dtype=float)\n        mi = q["minutes_played"].to_numpy(dtype=float)\n        m = len(q)\n\n        idx = rng.integers(0, m, size=(B, m))\n        total_v = va[idx].sum(axis=1)\n        total_m = mi[idx].sum(axis=1)\n\n        V[:, j] = (\n            90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n        )\n\n    if not np.isfinite(V).all():\n        raise RuntimeError("Non-finite bootstrap VAEP/90 values detected.")\n\n    # Rank every bootstrap leaderboard across all eligible players.\n    rank_draws = (\n        pd.DataFrame(V)\n        .rank(axis=1, ascending=False, method="average")\n        .to_numpy(dtype=float)\n    )\n\n    return player_ids, V, rank_draws\n\n\ndef pairwise_probability_matrix(V):\n    V = np.asarray(V, dtype=float)\n    K = V.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in tqdm(range(K), desc="Pairwise probabilities", unit="player"):\n        for j in range(i + 1, K):\n            d = V[:, i] - V[:, j]\n            p = float(\n                np.mean(\n                    np.where(\n                        d > 0,\n                        1.0,\n                        np.where(d == 0, 0.5, 0.0),\n                    )\n                )\n            )\n            P[i, j] = p\n            P[j, i] = 1.0 - p\n\n    return P\n\n\n# ================================================================\n# STEP 3 — PCS / Bradley-Terry consensus\n# ================================================================\n\ndef fit_pairwise_consensus(P, ridge=BT_RIDGE):\n    P = np.asarray(P, dtype=float)\n    K = P.shape[0]\n\n    if K == 1:\n        return np.array([0.0]), {\n            "success": True,\n            "pairwise_mae": 0.0,\n            "pairwise_rmse": 0.0,\n            "objective": 0.0,\n        }\n\n    ii, jj = np.triu_indices(K, k=1)\n    target = P[ii, jj]\n    valid = np.isfinite(target)\n    ii, jj, target = ii[valid], jj[valid], target[valid]\n\n    def loss_grad(s):\n        z = s[ii] - s[jj]\n        losses = np.logaddexp(0.0, z) - target * z\n        loss = float(np.mean(losses) + ridge * np.mean(s ** 2))\n\n        err = (expit(z) - target) / len(target)\n        grad = np.zeros_like(s)\n        np.add.at(grad, ii, err)\n        np.add.at(grad, jj, -err)\n        grad += (2.0 * ridge / K) * s\n\n        return loss, grad\n\n    result = minimize(\n        fun=lambda s: loss_grad(s)[0],\n        x0=np.zeros(K, dtype=float),\n        jac=lambda s: loss_grad(s)[1],\n        method="L-BFGS-B",\n        options={"maxiter": 1000, "ftol": 1e-12},\n    )\n\n    score = result.x - np.mean(result.x)\n    pred = expit(score[ii] - score[jj])\n\n    return score, {\n        "success": bool(result.success),\n        "pairwise_mae": float(np.mean(np.abs(pred - target))),\n        "pairwise_rmse": float(np.sqrt(np.mean((pred - target) ** 2))),\n        "objective": float(result.fun),\n    }\n\n\n# ================================================================\n# STEP 4 — Full static VAEP vs interval vs RAR table\n# ================================================================\n\ndef build_static_ranking_table(pm, cohort):\n    player_ids, V, rank_draws = bootstrap_full_cohort(\n        pm,\n        cohort,\n        B=FULL_BOOTSTRAP_B,\n        seed=SEED + 100,\n    )\n\n    P = pairwise_probability_matrix(V)\n    pcs, fit_info = fit_pairwise_consensus(P)\n\n    rar = ranks_desc(pcs)\n    K = len(player_ids)\n\n    q025_rank = np.quantile(rank_draws, 0.025, axis=0)\n    q50_rank = np.quantile(rank_draws, 0.50, axis=0)\n    q975_rank = np.quantile(rank_draws, 0.975, axis=0)\n\n    rows = []\n\n    cohort_ix = cohort.set_index("player_id")\n\n    for i, pid in enumerate(player_ids):\n        p_i = np.delete(P[i, :], i)\n\n        strongly_above = int(\n            np.sum(p_i <= (1.0 - PAIRWISE_SUPPORT))\n        )\n        strongly_below = int(\n            np.sum(p_i >= PAIRWISE_SUPPORT)\n        )\n\n        supported_low = 1 + strongly_above\n        supported_high = K - strongly_below\n\n        resolved = (\n            (p_i >= PAIRWISE_SUPPORT)\n            | (p_i <= (1.0 - PAIRWISE_SUPPORT))\n        )\n\n        c = cohort_ix.loc[pid]\n\n        rows.append({\n            "player_id": pid,\n            "player_name": c["player_name"],\n            "is_goalkeeper": bool(c["is_goalkeeper"]),\n            "matches": int(c["total_matches"]),\n            "minutes": float(c["total_minutes"]),\n            "actions": int(c["total_actions"]),\n            "VAEP90": float(c["vaep90"]),\n            "VAEP_rank": int(c["raw_VAEP_rank"]),\n            "bootstrap_median_rank": float(q50_rank[i]),\n            "bootstrap_rank_low_2.5pct": float(q025_rank[i]),\n            "bootstrap_rank_high_97.5pct": float(q975_rank[i]),\n            "bootstrap_rank_interval_width": float(q975_rank[i] - q025_rank[i]),\n            "supported_rank_low": int(supported_low),\n            "supported_rank_high": int(supported_high),\n            "supported_rank_width": int(supported_high - supported_low),\n            "pairwise_resolution_rate": float(np.mean(resolved)),\n            "PCS": float(pcs[i]),\n            "RAR": int(rar[i]),\n            "RAR_minus_VAEP_rank": int(rar[i] - c["raw_VAEP_rank"]),\n            "absolute_rank_shift": int(abs(rar[i] - c["raw_VAEP_rank"])),\n        })\n\n    table = pd.DataFrame(rows)\n\n    # Raw ordering and RAR ordering are both preserved in the same table.\n    table = table.sort_values(\n        ["VAEP_rank", "RAR"]\n    ).reset_index(drop=True)\n\n    # Overall diagnostics\n    median_interval_width = float(\n        table["bootstrap_rank_interval_width"].median()\n    )\n    median_supported_width = float(\n        table["supported_rank_width"].median()\n    )\n    changed_fraction = float(\n        np.mean(table["VAEP_rank"] != table["RAR"])\n    )\n    changed_count = int(\n        np.sum(table["VAEP_rank"] != table["RAR"])\n    )\n\n    pair_i, pair_j = np.triu_indices(K, k=1)\n    pair_probs = P[pair_i, pair_j]\n\n    supported_pair_fraction = float(\n        np.mean(\n            (pair_probs >= PAIRWISE_SUPPORT)\n            | (pair_probs <= (1.0 - PAIRWISE_SUPPORT))\n        )\n    )\n\n    diagnostics = pd.DataFrame([{\n        "dataset": DATASET_LABEL,\n        "eligible_players": K,\n        "bootstrap_draws": FULL_BOOTSTRAP_B,\n        "median_95pct_rank_interval_width": median_interval_width,\n        "median_supported_rank_band_width": median_supported_width,\n        "supported_pair_fraction_90pct": supported_pair_fraction,\n        "players_reordered_by_RAR": changed_count,\n        "fraction_players_reordered_by_RAR": changed_fraction,\n        "median_absolute_RAR_shift": float(table["absolute_rank_shift"].median()),\n        "max_absolute_RAR_shift": int(table["absolute_rank_shift"].max()),\n        "BT_fit_success": fit_info["success"],\n        "BT_pairwise_mae": fit_info["pairwise_mae"],\n        "BT_pairwise_rmse": fit_info["pairwise_rmse"],\n    }])\n\n    # Pairwise output file.\n    pair_rows = []\n    names = cohort_ix["player_name"].to_dict()\n\n    for i in range(K):\n        for j in range(i + 1, K):\n            p = float(P[i, j])\n            if p >= PAIRWISE_SUPPORT:\n                status = "i_above_j_supported"\n            elif p <= (1.0 - PAIRWISE_SUPPORT):\n                status = "j_above_i_supported"\n            else:\n                status = "unresolved"\n\n            pair_rows.append({\n                "player_i_id": player_ids[i],\n                "player_i_name": names[player_ids[i]],\n                "player_j_id": player_ids[j],\n                "player_j_name": names[player_ids[j]],\n                "P_i_above_j": p,\n                "support_status": status,\n                "BT_pred_P_i_above_j": float(expit(pcs[i] - pcs[j])),\n            })\n\n    pairs = pd.DataFrame(pair_rows)\n\n    return table, diagnostics, pairs\n\n\n# ================================================================\n# STEP 5 — Independent raw VAEP vs RAR reproducibility stress test\n# ================================================================\n\ndef bootstrap_fixed_sample(q, B, rng):\n    q = q.reset_index(drop=True)\n    va = q["vaep"].to_numpy(dtype=float)\n    mi = q["minutes_played"].to_numpy(dtype=float)\n    n = len(q)\n\n    idx = rng.integers(0, n, size=(B, n))\n    total_v = va[idx].sum(axis=1)\n    total_m = mi[idx].sum(axis=1)\n\n    return 90.0 * total_v / np.where(total_m > 0, total_m, np.nan)\n\n\ndef rar_from_draws(draws):\n    P = pairwise_probability_matrix_fast(draws)\n    pcs, fit_info = fit_pairwise_consensus(P)\n    rar = ranks_desc(pcs)\n    return rar, fit_info\n\n\ndef pairwise_probability_matrix_fast(V):\n    """\n    Same calculation as the full matrix, without progress bars.\n    Used inside repeated validation loops.\n    """\n    V = np.asarray(V, dtype=float)\n    K = V.shape[1]\n    P = np.full((K, K), 0.5, dtype=float)\n\n    for i in range(K):\n        xi = V[:, i]\n        for j in range(i + 1, K):\n            d = xi - V[:, j]\n            p = float(\n                np.mean(\n                    np.where(\n                        d > 0,\n                        1.0,\n                        np.where(d == 0, 0.5, 0.0),\n                    )\n                )\n            )\n            P[i, j] = p\n            P[j, i] = 1.0 - p\n\n    return P\n\n\ndef run_independent_reproducibility(pm, cohort):\n    validation_cohort = cohort[\n        cohort["total_matches"] >= VALIDATION_MIN_MATCHES\n    ].copy()\n\n    player_ids = validation_cohort["player_id"].tolist()\n\n    print("\\n" + "=" * 96)\n    print("INDEPENDENT REPRODUCIBILITY STRESS TEST")\n    print("=" * 96)\n    print("Fixed validation cohort size:", len(player_ids))\n    print("Minimum observed matches in fixed cohort:", int(validation_cohort["total_matches"].min()))\n    print("n values:", VALIDATION_N_VALUES)\n    print("repetitions per n:", VALIDATION_REPS)\n    print("inner bootstrap draws per sample:", INNER_BOOTSTRAP_B)\n\n    if len(player_ids) < 50:\n        print(\n            "WARNING: validation cohort has fewer than 50 players. "\n            "The test remains valid but is less of a large-leaderboard stress test."\n        )\n\n    min_m = int(validation_cohort["total_matches"].min())\n    invalid_n = [n for n in VALIDATION_N_VALUES if 2 * n > min_m]\n    if invalid_n:\n        raise RuntimeError(\n            f"Validation n values {invalid_n} are impossible because the fixed cohort "\n            f"has a minimum of only {min_m} matches."\n        )\n\n    grouped = {\n        pid: (\n            q.sort_values(["game_date", "game_id"])\n            .reset_index(drop=True)\n        )\n        for pid, q in pm[pm["player_id"].isin(player_ids)].groupby("player_id")\n    }\n\n    rng_master = np.random.default_rng(SEED + 7000)\n    raw_rows = []\n\n    for n in VALIDATION_N_VALUES:\n        print(f"\\nTesting n={n} matches per independent sample...")\n\n        for rep in tqdm(\n            range(VALIDATION_REPS),\n            desc=f"n={n}",\n            unit="rep",\n        ):\n            K = len(player_ids)\n            raw_A = np.empty(K, dtype=float)\n            raw_B = np.empty(K, dtype=float)\n            draws_A = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n            draws_B = np.empty((INNER_BOOTSTRAP_B, K), dtype=float)\n\n            for j, pid in enumerate(player_ids):\n                q = grouped[pid]\n\n                chosen = rng_master.choice(\n                    len(q),\n                    size=2 * n,\n                    replace=False,\n                )\n\n                qA = q.iloc[chosen[:n]].copy()\n                qB = q.iloc[chosen[n:]].copy()\n\n                raw_A[j] = vaep90(qA)\n                raw_B[j] = vaep90(qB)\n\n                rng_A = np.random.default_rng(\n                    SEED + 10_000_000 + n * 100_000 + rep * 1000 + j * 2\n                )\n                rng_B = np.random.default_rng(\n                    SEED + 20_000_000 + n * 100_000 + rep * 1000 + j * 2 + 1\n                )\n\n                draws_A[:, j] = bootstrap_fixed_sample(\n                    qA, INNER_BOOTSTRAP_B, rng_A\n                )\n                draws_B[:, j] = bootstrap_fixed_sample(\n                    qB, INNER_BOOTSTRAP_B, rng_B\n                )\n\n            raw_rank_A = ranks_desc(raw_A)\n            raw_rank_B = ranks_desc(raw_B)\n\n            rar_A, fit_A = rar_from_draws(draws_A)\n            rar_B, fit_B = rar_from_draws(draws_B)\n\n            rho_raw = float(\n                spearmanr(raw_rank_A, raw_rank_B).statistic\n            )\n            rho_rar = float(\n                spearmanr(rar_A, rar_B).statistic\n            )\n\n            raw_rows.append({\n                "n_matches_per_sample": n,\n                "rep": rep,\n                "players": K,\n                "raw_VAEP_spearman": rho_raw,\n                "RAR_spearman": rho_rar,\n                "RAR_minus_raw": rho_rar - rho_raw,\n                "RAR_better": int(rho_rar > rho_raw),\n                "RAR_equal": int(np.isclose(rho_rar, rho_raw)),\n                "RAR_worse": int(rho_rar < rho_raw),\n                "BT_A_success": bool(fit_A["success"]),\n                "BT_B_success": bool(fit_B["success"]),\n            })\n\n    raw = pd.DataFrame(raw_rows)\n\n    summary_rows = []\n\n    for n, q in raw.groupby("n_matches_per_sample"):\n        delta = q["RAR_minus_raw"].to_numpy(dtype=float)\n        nonzero = delta[~np.isclose(delta, 0.0)]\n\n        if len(nonzero) == 0:\n            p_value = 1.0\n        else:\n            try:\n                p_value = float(\n                    wilcoxon(\n                        nonzero,\n                        alternative="two-sided",\n                        zero_method="wilcox",\n                    ).pvalue\n                )\n            except Exception:\n                p_value = np.nan\n\n        summary_rows.append({\n            "n_matches_per_sample": int(n),\n            "players": int(q["players"].iloc[0]),\n            "reps": len(q),\n            "median_raw_VAEP_spearman": float(q["raw_VAEP_spearman"].median()),\n            "median_RAR_spearman": float(q["RAR_spearman"].median()),\n            "median_RAR_minus_raw": float(np.median(delta)),\n            "mean_RAR_minus_raw": float(np.mean(delta)),\n            "q025_RAR_minus_raw": float(np.quantile(delta, 0.025)),\n            "q975_RAR_minus_raw": float(np.quantile(delta, 0.975)),\n            "fraction_RAR_better": float(np.mean(delta > 0)),\n            "fraction_equal": float(np.mean(np.isclose(delta, 0.0))),\n            "fraction_RAR_worse": float(np.mean(delta < 0)),\n            "paired_wilcoxon_p": p_value,\n        })\n\n    summary = pd.DataFrame(summary_rows)\n\n    overall = pd.DataFrame([{\n        "dataset": DATASET_LABEL,\n        "fixed_validation_players": len(player_ids),\n        "validation_minimum_matches_required": VALIDATION_MIN_MATCHES,\n        "tested_n_values": ",".join(str(x) for x in VALIDATION_N_VALUES),\n        "reps_per_n": VALIDATION_REPS,\n        "inner_bootstrap_draws": INNER_BOOTSTRAP_B,\n        "pooled_median_RAR_minus_raw": float(raw["RAR_minus_raw"].median()),\n        "pooled_mean_RAR_minus_raw": float(raw["RAR_minus_raw"].mean()),\n        "pooled_fraction_RAR_better": float(np.mean(raw["RAR_minus_raw"] > 0)),\n        "pooled_fraction_equal": float(np.mean(np.isclose(raw["RAR_minus_raw"], 0.0))),\n        "pooled_fraction_RAR_worse": float(np.mean(raw["RAR_minus_raw"] < 0)),\n        "n_points_RAR_higher_median": int(\n            np.sum(summary["median_RAR_spearman"] > summary["median_raw_VAEP_spearman"])\n        ),\n        "n_points_equal_median": int(\n            np.sum(\n                np.isclose(\n                    summary["median_RAR_spearman"],\n                    summary["median_raw_VAEP_spearman"],\n                )\n            )\n        ),\n        "n_points_RAR_lower_median": int(\n            np.sum(summary["median_RAR_spearman"] < summary["median_raw_VAEP_spearman"])\n        ),\n    }])\n\n    return validation_cohort, raw, summary, overall\n\n\n# ================================================================\n# RUN COMPLETE ANALYSIS\n# ================================================================\n\npm, cohort, model_metrics, build_meta = build_pl_player_match()\n\nstatic_table, static_diagnostics, pairwise_table = build_static_ranking_table(\n    pm,\n    cohort,\n)\n\n# Save full tables.\nstatic_table.to_csv(\n    OUT_ROOT / "01_FULL_VAEP_vs_RAR_with_intervals.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["RAR", "VAEP_rank"]\n).to_csv(\n    OUT_ROOT / "02_FULL_RAR_ordered_leaderboard.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["VAEP_rank", "RAR"]\n).to_csv(\n    OUT_ROOT / "03_FULL_VAEP_ordered_leaderboard.csv",\n    index=False,\n)\n\nstatic_table.sort_values(\n    ["absolute_rank_shift", "VAEP_rank"],\n    ascending=[False, True],\n).to_csv(\n    OUT_ROOT / "04_players_with_largest_RAR_changes.csv",\n    index=False,\n)\n\npairwise_table.to_csv(\n    OUT_ROOT / "05_pairwise_probabilities.csv",\n    index=False,\n)\n\nstatic_diagnostics.to_csv(\n    OUT_ROOT / "06_static_diagnostics.csv",\n    index=False,\n)\n\nprint("\\n" + "=" * 96)\nprint("STATIC PREMIER LEAGUE RESULT")\nprint("=" * 96)\nprint(static_diagnostics.round(4).to_string(index=False))\n\nprint("\\nTop 30 by original VAEP rank:")\nprint(\n    static_table.sort_values("VAEP_rank")[\n        [\n            "VAEP_rank",\n            "player_name",\n            "VAEP90",\n            "bootstrap_rank_low_2.5pct",\n            "bootstrap_rank_high_97.5pct",\n            "supported_rank_low",\n            "supported_rank_high",\n            "PCS",\n            "RAR",\n            "RAR_minus_VAEP_rank",\n        ]\n    ]\n    .head(30)\n    .round(4)\n    .to_string(index=False)\n)\n\nprint("\\nTop 30 by RAR:")\nprint(\n    static_table.sort_values(["RAR", "VAEP_rank"])[\n        [\n            "RAR",\n            "player_name",\n            "PCS",\n            "VAEP_rank",\n            "VAEP90",\n            "bootstrap_rank_low_2.5pct",\n            "bootstrap_rank_high_97.5pct",\n            "supported_rank_low",\n            "supported_rank_high",\n            "RAR_minus_VAEP_rank",\n        ]\n    ]\n    .head(30)\n    .round(4)\n    .to_string(index=False)\n)\n\nprint("\\nLargest absolute RAR changes:")\nprint(\n    static_table.sort_values(\n        ["absolute_rank_shift", "VAEP_rank"],\n        ascending=[False, True],\n    )[\n        [\n            "player_name",\n            "VAEP_rank",\n            "RAR",\n            "RAR_minus_VAEP_rank",\n            "bootstrap_rank_low_2.5pct",\n            "bootstrap_rank_high_97.5pct",\n            "supported_rank_low",\n            "supported_rank_high",\n        ]\n    ]\n    .head(30)\n    .round(4)\n    .to_string(index=False)\n)\n\n# Independent validation.\nvalidation_cohort, validation_raw, validation_summary, validation_overall = (\n    run_independent_reproducibility(pm, cohort)\n)\n\nvalidation_cohort.to_csv(\n    OUT_ROOT / "07_fixed_validation_cohort_30plus_matches.csv",\n    index=False,\n)\n\nvalidation_raw.to_csv(\n    OUT_ROOT / "08_RAR_vs_raw_validation_replicates.csv",\n    index=False,\n)\n\nvalidation_summary.to_csv(\n    OUT_ROOT / "09_RAR_vs_raw_validation_by_n.csv",\n    index=False,\n)\n\nvalidation_overall.to_csv(\n    OUT_ROOT / "10_RAR_vs_raw_validation_overall.csv",\n    index=False,\n)\n\nprint("\\n" + "=" * 96)\nprint("RAW VAEP VS RAR REPRODUCIBILITY")\nprint("=" * 96)\nprint(validation_summary.round(4).to_string(index=False))\n\nprint("\\nOverall validation:")\nprint(validation_overall.round(4).to_string(index=False))\n\n\n# ================================================================\n# Figures\n# ================================================================\n\n# Figure 1: top 50 raw VAEP rank vs RAR.\ntop50_ids = (\n    static_table.sort_values("VAEP_rank")\n    .head(50)["player_id"]\n    .tolist()\n)\nq50 = static_table[\n    static_table["player_id"].isin(top50_ids)\n].copy().sort_values("VAEP_rank")\n\nfig, ax = plt.subplots(figsize=(10, 6))\nax.scatter(q50["VAEP_rank"], q50["RAR"])\nmax_rank = max(q50["VAEP_rank"].max(), q50["RAR"].max())\nax.plot([1, max_rank], [1, max_rank], linestyle="--")\nax.set_xlabel("Original VAEP rank")\nax.set_ylabel("RAR")\nax.set_title("Premier League 2015/16: top-50 VAEP rank vs RAR")\nax.invert_xaxis()\nax.invert_yaxis()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "11_top50_VAEP_rank_vs_RAR.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 2: interval width by original rank.\nfig, ax = plt.subplots(figsize=(10, 6))\nax.scatter(\n    static_table["VAEP_rank"],\n    static_table["bootstrap_rank_interval_width"],\n)\nax.set_xlabel("Original VAEP rank")\nax.set_ylabel("95% bootstrap rank-interval width")\nax.set_title("Premier League 2015/16 ranking uncertainty")\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "12_bootstrap_interval_width_by_rank.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 3: independent reproducibility.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.plot(\n    validation_summary["n_matches_per_sample"],\n    validation_summary["median_raw_VAEP_spearman"],\n    marker="o",\n    label="Raw VAEP rank",\n)\nax.plot(\n    validation_summary["n_matches_per_sample"],\n    validation_summary["median_RAR_spearman"],\n    marker="o",\n    label="RAR",\n)\nax.axhline(0.0, linestyle=":")\nax.set_xlabel("Matches per independent sample")\nax.set_ylabel("Median Spearman reproducibility")\nax.set_title("Premier League: raw VAEP vs RAR reproducibility")\nax.set_ylim(-1, 1)\nax.legend()\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "13_raw_VAEP_vs_RAR_reproducibility.png",\n    dpi=180,\n)\nplt.close(fig)\n\n# Figure 4: delta.\nfig, ax = plt.subplots(figsize=(8, 5.5))\nax.plot(\n    validation_summary["n_matches_per_sample"],\n    validation_summary["median_RAR_minus_raw"],\n    marker="o",\n)\nax.axhline(0.0, linestyle="--")\nax.set_xlabel("Matches per independent sample")\nax.set_ylabel("Median Spearman gain (RAR - raw VAEP)")\nax.set_title("Does RAR improve ranking reproducibility?")\nplt.tight_layout()\nfig.savefig(\n    OUT_ROOT / "14_RAR_reproducibility_gain.png",\n    dpi=180,\n)\nplt.close(fig)\n\n\n# ================================================================\n# Paper-facing summary\n# ================================================================\n\ndiag = static_diagnostics.iloc[0]\nov = validation_overall.iloc[0]\n\nsummary_lines = [\n    "ReliableVAEP Premier League 2015/16 PCS/RAR stress test",\n    "=" * 70,\n    "",\n    f"Eligible players: {int(diag[\'eligible_players\'])}",\n    f"Median 95% bootstrap rank-interval width: {diag[\'median_95pct_rank_interval_width\']:.2f}",\n    f"Median supported rank-band width: {diag[\'median_supported_rank_band_width\']:.2f}",\n    f"Supported pair fraction at 90%: {100*diag[\'supported_pair_fraction_90pct\']:.1f}%",\n    f"Players reordered by RAR: {int(diag[\'players_reordered_by_RAR\'])} "\n    f"({100*diag[\'fraction_players_reordered_by_RAR\']:.1f}%)",\n    f"Median absolute RAR shift: {diag[\'median_absolute_RAR_shift\']:.2f} ranks",\n    f"Maximum absolute RAR shift: {int(diag[\'max_absolute_RAR_shift\'])} ranks",\n    "",\n    "Independent reproducibility test:",\n    f"Fixed >=30-match cohort: {int(ov[\'fixed_validation_players\'])} players",\n    f"Pooled median Spearman gain (RAR - raw): {ov[\'pooled_median_RAR_minus_raw\']:.4f}",\n    f"Pooled mean Spearman gain (RAR - raw): {ov[\'pooled_mean_RAR_minus_raw\']:.4f}",\n    f"Replicates RAR better/equal/worse: "\n    f"{ov[\'pooled_fraction_RAR_better\']:.3f} / "\n    f"{ov[\'pooled_fraction_equal\']:.3f} / "\n    f"{ov[\'pooled_fraction_RAR_worse\']:.3f}",\n    "",\n    "Interpretation rule:",\n    "RAR may be described as a more reproducible reliability-adjusted ranking only "\n    "if the independent non-overlapping-match test shows a consistent positive gain. "\n    "If not, keep RAR as a descriptive consensus ordering and retain the original VAEP "\n    "rank, bootstrap interval, and supported rank band as the primary reliability outputs.",\n]\n\nsummary_text = "\\n".join(summary_lines)\n(OUT_ROOT / "15_paper_facing_summary.txt").write_text(\n    summary_text,\n    encoding="utf-8",\n)\n\nprint("\\n" + summary_text)\n\n# Final ZIP.\nzip_path = shutil.make_archive(\n    str(ROOT / "ReliableVAEP_PL1516_PCS_RAR_STRESS_TEST_RESULTS"),\n    "zip",\n    root_dir=OUT_ROOT,\n)\n\nprint("\\n" + "=" * 96)\nprint("ANALYSIS COMPLETE")\nprint("=" * 96)\nprint("Results folder:", OUT_ROOT)\nprint("Results ZIP:", zip_path)\n'

SCRIPT_PATH.write_text(
    ANALYSIS_SCRIPT,
    encoding="utf-8",
)

print("Analysis script written to:", SCRIPT_PATH)
print("Script size:", SCRIPT_PATH.stat().st_size, "bytes")


Analysis script written to: /kaggle/working/reliablevaep_pl1516_pcs_rar_stress_test.py
Script size: 46541 bytes


In [4]:

# ================================================================
# CELL 3 — RUN THE COMPLETE ANALYSIS
# ================================================================

import os
import subprocess
from pathlib import Path

ENV_PY = Path("/kaggle/working/rv_vaep_env/bin/python")
SCRIPT_PATH = Path(
    "/kaggle/working/reliablevaep_pl1516_pcs_rar_stress_test.py"
)

if not ENV_PY.exists():
    raise RuntimeError(
        "Run Cell 1 first. The isolated Kaggle VAEP environment does not exist."
    )

if not SCRIPT_PATH.exists():
    raise RuntimeError(
        "Run Cell 2 first. The analysis script does not exist."
    )

child_env = os.environ.copy()
child_env["MPLBACKEND"] = "Agg"
child_env.pop("MATPLOTLIBRC", None)

print("Running Premier League PCS/RAR stress test...")
print("Analysis Python:", ENV_PY)
print()

process = subprocess.Popen(
    [str(ENV_PY), "-u", str(SCRIPT_PATH)],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=child_env,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Analysis exited with code {return_code}. "
        "The exact traceback is shown immediately above."
    )

print("\nPremier League PCS/RAR stress test finished successfully.")


Running Premier League PCS/RAR stress test...
Analysis Python: /kaggle/working/rv_vaep_env/bin/python

ReliableVAEP Premier League 2015/16 PCS/RAR stress test
Eligibility: >=5 matches, >=300 minutes, >=150 actions
Full-data bootstrap draws: 3500
Pairwise support threshold: 0.9
Validation fixed cohort: >= 30 matches
Validation n values: [5, 10, 15]
Validation repetitions per n: 200

Reference only: the earlier PL analysis had about 325 eligible players and a median 95% bootstrap rank-interval width of about 52. This notebook recomputes everything canonically and does NOT force those values.

StatsBomb PL matches found: 380

PL 2015/16: SPADL: 100%|██████████| 380/380 [14:28<00:00,  2.29s/match]

Canonical VAEP features: 100%|██████████| 380/380 [01:16<00:00,  4.95match/s]
Action rows=758,433, features=568, score rate=0.0109, concede rate=0.0021

Cross-fit model means:
fold                          3.0000
fit_matches                 243.0000
calibration_matches          61.0000
test_matc

In [5]:

# ================================================================
# CELL 4 — SHOW VAEP RANKING, INTERVALS, AND RAR RANKING
# ================================================================
#
# This cell deliberately uses only the Python standard library +
# IPython display, so it does not depend on Kaggle's host pandas.
# ================================================================

import csv
import html
from pathlib import Path
from IPython.display import display, HTML

ROOT = Path("/kaggle/working")
OUT = ROOT / "rv_pl1516_pcs_rar_outputs"

combined_csv = OUT / "01_FULL_VAEP_vs_RAR_with_intervals.csv"
diagnostics_csv = OUT / "06_static_diagnostics.csv"
validation_csv = OUT / "09_RAR_vs_raw_validation_by_n.csv"

if not combined_csv.exists():
    raise FileNotFoundError(
        "The ranking output does not exist yet. "
        "Cell 3 must finish successfully first."
    )


def read_csv_dict(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


rows = read_csv_dict(combined_csv)


def as_float(x):
    try:
        return float(x)
    except Exception:
        return float("nan")


def format_rank(x):
    v = as_float(x)
    if v == int(v):
        return str(int(v))
    return f"{v:.1f}"


def render_table(title, records, columns):
    parts = [
        f"<h3>{html.escape(title)}</h3>",
        '<div style="overflow-x:auto; max-height:650px; overflow-y:auto;">',
        '<table style="border-collapse:collapse; width:100%; font-size:13px;">',
        "<thead><tr>",
    ]

    for key, label in columns:
        parts.append(
            f'<th style="position:sticky;top:0;background:#f3f4f6;'
            f'border:1px solid #ddd;padding:6px;text-align:left;">'
            f'{html.escape(label)}</th>'
        )
    parts.append("</tr></thead><tbody>")

    for r in records:
        parts.append("<tr>")
        for key, label in columns:
            value = r.get(key, "")
            if key in {
                "VAEP90",
                "PCS",
                "pairwise_resolution_rate",
            }:
                try:
                    value = f"{float(value):.4f}"
                except Exception:
                    pass
            elif "rank" in key.lower() or key in {"RAR", "VAEP_rank"}:
                try:
                    value = format_rank(value)
                except Exception:
                    pass

            parts.append(
                f'<td style="border:1px solid #ddd;padding:5px;">'
                f'{html.escape(str(value))}</td>'
            )
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    display(HTML("".join(parts)))


# Full table columns requested by the user.
columns = [
    ("VAEP_rank", "VAEP rank"),
    ("player_name", "Player"),
    ("VAEP90", "VAEP/90"),
    ("bootstrap_rank_low_2.5pct", "95% interval low"),
    ("bootstrap_rank_high_97.5pct", "95% interval high"),
    ("supported_rank_low", "Supported low"),
    ("supported_rank_high", "Supported high"),
    ("PCS", "PCS"),
    ("RAR", "RAR"),
    ("RAR_minus_VAEP_rank", "RAR - VAEP rank"),
]

vaep_order = sorted(rows, key=lambda r: as_float(r["VAEP_rank"]))
rar_order = sorted(rows, key=lambda r: (as_float(r["RAR"]), as_float(r["VAEP_rank"])))
changed_order = sorted(
    rows,
    key=lambda r: (-as_float(r["absolute_rank_shift"]), as_float(r["VAEP_rank"])),
)

render_table(
    "Top 50 by original VAEP ranking",
    vaep_order[:50],
    columns,
)

render_table(
    "Top 50 by Reliability-Adjusted Rank (RAR)",
    rar_order[:50],
    [
        ("RAR", "RAR"),
        ("player_name", "Player"),
        ("PCS", "PCS"),
        ("VAEP_rank", "VAEP rank"),
        ("VAEP90", "VAEP/90"),
        ("bootstrap_rank_low_2.5pct", "95% interval low"),
        ("bootstrap_rank_high_97.5pct", "95% interval high"),
        ("supported_rank_low", "Supported low"),
        ("supported_rank_high", "Supported high"),
        ("RAR_minus_VAEP_rank", "RAR - VAEP rank"),
    ],
)

render_table(
    "30 players with the largest VAEP → RAR rank changes",
    changed_order[:30],
    [
        ("player_name", "Player"),
        ("VAEP_rank", "VAEP rank"),
        ("RAR", "RAR"),
        ("RAR_minus_VAEP_rank", "Rank change"),
        ("bootstrap_rank_low_2.5pct", "95% interval low"),
        ("bootstrap_rank_high_97.5pct", "95% interval high"),
        ("supported_rank_low", "Supported low"),
        ("supported_rank_high", "Supported high"),
    ],
)

if diagnostics_csv.exists():
    diagnostics = read_csv_dict(diagnostics_csv)
    render_table(
        "Static ranking diagnostics",
        diagnostics,
        [(k, k.replace("_", " ")) for k in diagnostics[0].keys()],
    )

if validation_csv.exists():
    validation = read_csv_dict(validation_csv)
    render_table(
        "Independent reproducibility: raw VAEP vs RAR",
        validation,
        [
            ("n_matches_per_sample", "Matches/sample"),
            ("players", "Players"),
            ("median_raw_VAEP_spearman", "Median raw VAEP ρ"),
            ("median_RAR_spearman", "Median RAR ρ"),
            ("median_RAR_minus_raw", "Median Δρ"),
            ("fraction_RAR_better", "RAR better"),
            ("fraction_equal", "Equal"),
            ("fraction_RAR_worse", "RAR worse"),
            ("paired_wilcoxon_p", "Wilcoxon p"),
        ],
    )

print("\nFull combined ranking table:")
print(combined_csv)

print("\nResults ZIP:")
print(ROOT / "ReliableVAEP_PL1516_PCS_RAR_STRESS_TEST_RESULTS.zip")


VAEP rank,Player,VAEP/90,95% interval low,95% interval high,Supported low,Supported high,PCS,RAR,RAR - VAEP rank
1,Kelechi Promise Iheanacho,0.6448,1,157.1,1,45,4.5450,4,3
2,Dimitri Payet,0.5578,1,22,1,18,5.7436,1,-1
3,Riyad Mahrez,0.5152,1,48,1,38,5.0084,2,-1
4,Sergio Leonel Agüero del Castillo,0.5132,1,115,1,65,4.3271,5,1
5,Mesut Özil,0.4795,3,37,1,40,4.9136,3,-2
6,Mauro Matías Zárate,0.4604,1,200,1,98,3.6857,12,6
7,Daniel Andre Sturridge,0.4274,2,203.1,1,107,3.5696,14,7
8,Philippe Coutinho Correia,0.4180,2,145,1,87,3.7718,8,0
9,Alexis Alejandro Sánchez Sánchez,0.4138,2,144.5,1,89,3.7321,9,0
10,Dušan Tadić,0.4084,3,94,1,78,3.9478,6,-4


RAR,Player,PCS,VAEP rank,VAEP/90,95% interval low,95% interval high,Supported low,Supported high,RAR - VAEP rank
1,Dimitri Payet,5.7436,2,0.5578,1,22,1,18,-1
2,Riyad Mahrez,5.0084,3,0.5152,1,48,1,38,-1
3,Mesut Özil,4.9136,5,0.4795,3,37,1,40,-2
4,Kelechi Promise Iheanacho,4.5450,1,0.6448,1,157.1,1,45,3
5,Sergio Leonel Agüero del Castillo,4.3271,4,0.5132,1,115,1,65,1
6,Dušan Tadić,3.9478,10,0.4084,3,94,1,78,-4
7,Francesc Fàbregas i Soler,3.8677,16,0.3720,10,61,2,79,-9
8,Philippe Coutinho Correia,3.7718,8,0.4180,2,145,1,87,0
9,Alexis Alejandro Sánchez Sánchez,3.7321,9,0.4138,2,144.5,1,89,0
10,Gerard Deulofeu Lázaro,3.7136,15,0.3730,7,89,2,87,-5


Player,VAEP rank,RAR,Rank change,95% interval low,95% interval high,Supported low,Supported high
Nathan Dyer,358,288,-70,4,423,6,422
Callum Wilson,94,156,62,2,415,2,409
Rolando Aarons,107,168,61,1,415,1,410
Eunan O''Kane,250,310,60,167,380,145,395
Daniel Nii Tackie Mensah Welbeck,110,167,57,2,418,2,411
Daniel William John Ings,23,77,54,1,403.5,1,377
Marcus Rashford,35,83,48,2,390,1,356
Alex Iwobi,40,85,45,3,388,2,341
Robert Kenedy Nunes do Nascimento,75,119,44,6,380,5,366
Connor Wickham,381,337,-44,22,419,49,420


dataset,eligible players,bootstrap draws,median 95pct rank interval width,median supported rank band width,supported pair fraction 90pct,players reordered by RAR,fraction players reordered by RAR,median absolute RAR shift,max absolute RAR shift,BT fit success,BT pairwise mae,BT pairwise rmse
Premier League 2015/16,423,3500,201.1,241,0.44822022789149946,397,0.9385342789598109,7.0,70,True,0.05022836529094978,0.07052306736575276


Matches/sample,Players,Median raw VAEP ρ,Median RAR ρ,Median Δρ,RAR better,Equal,RAR worse,Wilcoxon p
5,141,0.22483197054668436,0.24926714430378716,0.02220557386874436,0.865,0.0,0.135,2.278367673718281e-25
10,141,0.37783509568046575,0.3879853589622843,0.012569221214060788,0.775,0.0,0.225,3.752163817228538e-17
15,141,0.4834581959844171,0.4902928207543131,0.004831114345648341,0.675,0.0,0.325,1.0037518854396323e-07



Full combined ranking table:
/kaggle/working/rv_pl1516_pcs_rar_outputs/01_FULL_VAEP_vs_RAR_with_intervals.csv

Results ZIP:
/kaggle/working/ReliableVAEP_PL1516_PCS_RAR_STRESS_TEST_RESULTS.zip



## Main outputs

After the run finishes, the most important file is:

`/kaggle/working/rv_pl1516_pcs_rar_outputs/01_FULL_VAEP_vs_RAR_with_intervals.csv`

It contains the entire eligible Premier League leaderboard with:

**VAEP rank → 95% bootstrap interval → supported rank band → PCS → RAR**

Additional files:

- `02_FULL_RAR_ordered_leaderboard.csv` — full leaderboard sorted by RAR
- `03_FULL_VAEP_ordered_leaderboard.csv` — full leaderboard sorted by original VAEP
- `04_players_with_largest_RAR_changes.csv` — strongest reordering cases
- `06_static_diagnostics.csv` — eligible-player count, observed median interval width, fraction reordered, etc.
- `09_RAR_vs_raw_validation_by_n.csv` — independent reproducibility comparison
- `10_RAR_vs_raw_validation_overall.csv` — overall RAR validation
- `15_paper_facing_summary.txt` — conservative result summary

Complete ZIP:

`/kaggle/working/ReliableVAEP_PL1516_PCS_RAR_STRESS_TEST_RESULTS.zip`


In [ ]:

from pathlib import Path

zip_path = Path(
    "/kaggle/working/ReliableVAEP_PL1516_PCS_RAR_STRESS_TEST_RESULTS.zip"
)

if zip_path.exists():
    print("Results ZIP is ready:")
    print(zip_path)
else:
    print(
        "Results ZIP is not available yet. "
        "Run the complete analysis first."
    )
